In [1]:
# APCEMM Application of PCE with LRT
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
from numpy.linalg import eig
from scipy.interpolate import interp1d
import totalOrderMultiIndexSet
from sklearn.neighbors import KernelDensity
from numpy.polynomial.hermite_e import HermiteE
from matplotlib.lines import Line2D

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import cdsapi

plt.rcParams["figure.figsize"] = (10, 6)
import tqdm
import time

# For LRT
#!/usr/bin/python
import sys

import warnings
def fxn():
    warnings.warn("deprecated", DeprecationWarning)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fxn()

In [2]:
### FUNCTION LIBRARY ###

def PCE():
    
    print("Started!")
    print("--- %s minutes elapsed ---" % round(time.time()/60 - start_time/60,2))
    
    c, alpha_set = compute_c()
    multiindices = alpha_set.shape[0]
    
    print("Got c and alpha!")

    He_array = np.zeros((multiindices, mc_runs))
    u_evals = np.ones(timesteps)
    
    print("Started MC runs!")

    comparison_samples_matrix = get_samples_matrix(mc_runs)

    for i in tqdm.tqdm(range(mc_runs)): # for each MC run
        u_evals = np.vstack((u_evals, solve_u(comparison_samples_matrix)))
        
        for j in range(multiindices): # for each coefficient
            current_alpha = alpha_set[j, :] # look at one row at a time for all alpha describing a single coefficient
            samples_matrix_He = comparison_samples_matrix[i, :]
            He_array[j, i] = compute_He(samples_matrix_He, current_alpha)

    # Get rid of first row of ones
    true_output = u_evals[1:,:]

    # Solve for the Least Squares solution
    predicted_output = c.T @ He_array
    
    print("--- %s minutes elapsed ---" % round(time.time()/60 - start_time/60,2))
    
    return predicted_output.T, true_output, c, alpha_set

def compute_c():
    
    c_samples_matrix = get_samples_matrix(training_runs)
    
    alpha_set = get_alpha_set()
    print("alpha set shape: ", alpha_set.shape)

    c_multiindices = alpha_set.shape[0]

    V = np.zeros((training_runs, c_multiindices)) # Vandermonde Matrix: Timesteps x Degree of Polynomial

    for i in tqdm.tqdm(range(training_runs)):
        for j in range(c_multiindices):
            current_alpha = alpha_set[j, :]
            c_samples_matrix_He = c_samples_matrix[i, :]
            V[i, j] = compute_He(c_samples_matrix_He, current_alpha) / np.sqrt(np.product(sc.special.factorial(current_alpha)))
    
    f = solve_u(c_samples_matrix) # f should be same dimension as V (Training_runs x Degree of Polynomial)
    
    c = np.linalg.solve(V.T @ V, V.T @ f)
    
    return c, alpha_set

def dummy_func_eval(timesteps, samples_array):

    true_output = np.zeros_like(samples_array)
    for t in range(0, timesteps):
        true_output[:,t] = samples_array[:,t] + (np.sin(2*np.pi*(t+1)/timesteps))
    
    return true_output

def get_samples_matrix(runs):
    return np.random.normal(mean_Y, sigma_Y, size = (runs, timesteps)) # was 0,1

def get_alpha_set():
    return totalOrderMultiIndexSet.totalOrderMultiIndices(poly_dim, max_deg)

def compute_He(Z, alpha):
    res = 1
    for alpha_i, z_i in zip(alpha, Z):
        res = res * HermiteE.basis(deg = alpha_i)(z_i)
    return res

def solve_u(samples_matrix):
    
    u_sol = LRT_func_eval(timesteps, samples_matrix) # dummy_func_eval(timesteps, samples_matrix) #NOT SURE
    return u_sol

#-----------------------------------------------------------------------------------

def validate_PCE(samples_matrix, c, alpha_set):
    multiindices = alpha_set.shape[0]
    He_array = np.zeros((multiindices, validation_runs))
    u_evals = np.ones(timesteps)

    for i in tqdm.tqdm(range(validation_runs)): # for each MC run
        u_evals = np.vstack((u_evals, solve_u(samples_matrix)))
        
        for j in range(multiindices): # for each coefficient
            current_alpha = alpha_set[j, :] # look at one row at a time for all alpha describing a single coefficient
            samples_matrix_He = samples_matrix[i, :]
            He_array[j, i] = compute_He(samples_matrix_He, current_alpha)

    # Get rid of first row of ones
    true_output = u_evals[1:,:]

    # Solve for the Least Squares solution
    predicted_output = (c.T @ He_array).T

    return predicted_output, true_output

#-----------------------------------------------------------------------------------
### LRT FUNCTION LIBRARY ###

def LRT_func_eval(timesteps, samples_array):
    true_output = np.zeros_like(samples_array)
    num_samples = samples_array.shape[0] # check if correct
    for j in tqdm.tqdm(range(0, num_samples)):
        for t in range(0, timesteps):
            # Update contrail LWC
            sample = samples_array[j,t]
            updateOD(t, sample)
            contrailFluxRaw = contrailRF(attributes,"ice")
            contrailFlux = reformatResults(contrailFluxRaw)[-1]
            clearFluxRaw = clearskyRF(attributes)
            clearFlux = reformatResults(clearFluxRaw)[-1]
            true_output[j,t] = float(contrailFlux) - float(clearFlux)
        print("Contrail RF f(t): ", true_output[j,:])
            
    return true_output

def updateOD(timestep,sample):
    print("timestep: ", timestep)
    print("sample: ", sample)
    sample = (sample*10) + 25 # transform variance and mean of distribution
    if sample <= 0:
        sample = 0.0001
    attributes.ic_modify[0] = str(sample)


def updateInput(filepath, attributes, contrail, type):
    """
    Update the input file with the given attributes.

    Parameters:
    - filepath (str): The path of the input file to be updated.
    - attributes (object): An object containing the attributes to be written to the input file.
    - contrail (bool): A flag indicating whether the input file is for contrail simulation or not.

    Returns:
    None
    """
    file = open(filepath,"w")
    if contrail == True and type == "water":
        file.writelines(["rte_solver "+attributes.rte_solver[0]+"\n",
                            "source "+attributes.source[0]+"\n",
                            "sza "+attributes.sza[0]+"\n",
                            "wavelength "+attributes.wavelength[0]+"\n",
                            "mol_abs_param "+attributes.mol_abs_param[0]+"\n",
                            "umu "+attributes.umu[0]+"\n",
                            "output_user "+attributes.output_user[0]+"\n",
                            "zout "+attributes.zout[0]+"\n",
                            "output_process "+attributes.output_process[0]+"\n",
                            "atmosphere_file "+str(attributes.atmosphere_file[0])+"\n",
                            "wc_file 1D" +attributes.wc_file[0]+"\n",
                            "quiet"])
        file.close()
    elif contrail == True and type == "ice":
        file.writelines(["rte_solver "+attributes.rte_solver[0]+"\n",
                            "source "+attributes.source[0]+"\n",
                            "sza "+attributes.sza[0]+"\n",
                            "wavelength "+attributes.wavelength[0]+"\n",
                            "mol_abs_param "+attributes.mol_abs_param[0]+"\n",
                            "umu "+attributes.umu[0]+"\n",
                            "output_user "+attributes.output_user[0]+"\n",
                            "zout "+attributes.zout[0]+"\n",
                            "output_process "+attributes.output_process[0]+"\n",
                            "atmosphere_file "+str(attributes.atmosphere_file[0])+"\n", 
                            "ic_habit "+str(attributes.ic_habit[0])+"\n",
                            "ic_properties "+str(attributes.ic_properties[0])+"\n",
                            "ic_file 1D "+attributes.ic_file[0]+"\n",
                            "ic_modify tau set "+attributes.ic_modify[0]+"\n",
                            "quiet"])
        file.close()
    elif contrail == False and type == "clear": 
        file.writelines(["rte_solver "+attributes.rte_solver[0]+"\n",
                            "source "+attributes.source[0]+"\n",
                            "sza "+attributes.sza[0]+"\n",
                            "wavelength "+attributes.wavelength[0]+"\n",
                            "mol_abs_param "+attributes.mol_abs_param[0]+"\n",
                            "umu "+attributes.umu[0]+"\n",
                            "output_user "+attributes.output_user[0]+"\n",
                            "zout "+attributes.zout[0]+"\n",
                            "output_process "+attributes.output_process[0]+"\n",
                            "atmosphere_file "+str(attributes.atmosphere_file[0])+"\n",
                            "quiet\n"])
        file.close()
    
def clearskyRF(attributes):
    """
    Run the clear sky radiative forcing simulation.

    Parameters:
    - attributes (object): An object containing the attributes for the simulation.

    Returns:
    - LRToutput (list): A list containing the output of the simulation.
    """
    updateInput("/home/chinahg/GCresearch/contrailuncertainty/LRT/thermal-clear.in", attributes, False, "clear")
    LRToutput = !/home/chinahg/GCresearch/contrailuncertainty/LRT/uvspec < /home/chinahg/GCresearch/contrailuncertainty/LRT/thermal-clear.in # [X,X,X,net TOA flux]
    return LRToutput

def contrailRF(attributes, type):
    """
    Run the contrail radiative forcing simulation.

    Parameters:
    - attributes (object): An object containing the attributes for the simulation.

    Returns:
    - LRToutput (list): A list containing the output of the simulation.
    """
    updateInput("/home/chinahg/GCresearch/contrailuncertainty/LRT/thermal-cloud.in", attributes, True, type)
    LRToutput = !/home/chinahg/GCresearch/contrailuncertainty/LRT/uvspec < /home/chinahg/GCresearch/contrailuncertainty/LRT/thermal-cloud.in # [X,X,X,X,X,net TOA flux]
    return LRToutput

def reformatResults(resultsRaw):
    """
    Reformat the raw results.

    Parameters:
    - resultsRaw (list): A list containing the raw results.

    Returns:
    - li (list): A list containing the reformatted results.
    """
    string = str(resultsRaw[0].strip().replace("  ", " "))
    li = list(string.split(" ")) 
    return li

In [3]:
# Define constants
training_runs = 10 # Number of LRT runs for training
timesteps = 10 # Number of samples per LRT run (this is the number of timesteps per run)
max_deg = 2 # Maximum value of sum of degrees across the number of uncertain variables (AKA sum of each row of alpha_set must be less than or equal to the max_deg)
poly_dim = 1 # Number of random variables
mc_runs = 10 # Number of PCE runs

#FOR TESTING
mean_Y = 1
sigma_Y = 0.5 #squared value?

# Define LRT constants
# Set-up 
rte_solver = "disort" # 1D radiative transfer solver (DIScrete ORdinaTe solver)
source = "thermal" # Absorbing on the thermal spectrum (not solar)
sza = "0" # Solar zenith angle
wavelength = "2500 80000" # Wavelength range to compute over
mol_abs_param = "reptran fine" # spectral resolution (fine/medium)
umu = "1.0" # cosine of the viewing zenith angle
output_user = "edir eglo edn eup enet esum" # The direct, global, diffuse downward, 
                                            #and diffuse upward irradiance. Net is 
                                            #global - upward, sum is global + upward.
zout = "TOA" # Top of atmosphere (where total flux is calculated)
output_process = "integrate" # Integrate over wavelength
atmosphere_file = "midlatitude_summer" #, "midlatitude_winter", "subarctic_summer", "subarctic_winter", "tropics", "US-standard"] # Standard atmosphere type
ic_habit = "droxtal" #, "hollow-column", "rough-aggregate", "rosette-4", "rosette-6", "plate", "droxtal", "dendrite", "spheroid"]
ic_properties = "yang"
ic_file = "/home/chinahg/GCresearch/contrailuncertainty/LRT/ice.in"
wc_file = "/home/chinahg/GCresearch/contrailuncertainty/LRT/cloud.in"
ic_modify = "15"

inputs = {'rte_solver': [rte_solver], 'source': [source], 'sza': [sza], 'wavelength': [wavelength], 
                      'mol_abs_param': [mol_abs_param], 'umu': [umu], 'output_user': [output_user], 'zout': [zout], 
                      'output_process': [output_process], 'atmosphere_file': [atmosphere_file], 'ic_habit': [ic_habit], 
                      'ic_properties': [ic_properties], 'ic_file': [ic_file], 'wc_file': [wc_file], 'ic_modify': [ic_modify]}
attributes = pd.DataFrame(data = inputs)

In [4]:
start_time = time.time()

# Call PCE function
predicted_output, training_output, c, alpha = PCE()

Started!
--- 0.0 minutes elapsed ---
alpha set shape:  (3, 1)


  0%|          | 0/10 [00:00<?, ?it/s]

timestep:  0
sample:  1.5329156218812914


/tmp/ipykernel_842544/530026063.py:135: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  attributes.ic_modify[0] = str(sample)


timestep:  1
sample:  0.634726462730085
timestep:  2
sample:  0.40055620978994555
timestep:  3
sample:  2.431944508903754
timestep:  4
sample:  1.6620332810848015
timestep:  5
sample:  2.2385107905236654
timestep:  6
sample:  0.8606286750923449
timestep:  7
sample:  1.3211946330511526
timestep:  8
sample:  0.768209409265798
timestep:  9
sample:  0.6527628911440483


...

Contrail RF f(t):  [-114.1382 -114.0541 -112.1657 -113.0303 -114.4331 -114.0743 -113.8409
 -112.7489 -113.928  -114.4992]
timestep:  0
sample:  -0.01687559651192161
timestep:  1
sample:  0.8670137263882589
timestep:  2
sample:  1.8493299336717972
timestep:  3
sample:  2.5084208797122103
timestep:  4
sample:  1.2486821536978334
timestep:  5
sample:  0.3214368402974481
timestep:  6
sample:  0.9350752029175484
timestep:  7
sample:  1.1002640676031434
timestep:  8
sample:  0.7631698279064204
timestep:  9
sample:  1.162861032258295


 30%|███       | 3/10 [38:05<1:29:03, 763.33s/it]

Contrail RF f(t):  [-112.1811 -113.6816 -114.6345 -115.065  -114.111  -112.8664 -113.7651
 -113.9548 -113.5474 -114.0222]
timestep:  0
sample:  0.8134839613542558
timestep:  1
sample:  1.5070130528842474
timestep:  2
sample:  1.06969078920275
timestep:  3
sample:  0.2610053129261076
timestep:  4
sample:  0.13432764904945782
timestep:  5
sample:  1.0530655148030437
timestep:  6
sample:  0.6167158708990409
timestep:  7
sample:  0.8501771801515692
timestep:  8
sample:  1.2388711725802475
timestep:  9
sample:  0.9827832664312356


 40%|████      | 4/10 [50:58<1:16:43, 767.17s/it]

Contrail RF f(t):  [-113.6135 -114.3553 -113.921  -112.7563 -112.5091 -113.9024 -113.343
 -113.6604 -114.101  -113.8217]
timestep:  0
sample:  1.6442655078340485
timestep:  1
sample:  0.7034749970092391
timestep:  2
sample:  1.327160496575955
timestep:  3
sample:  1.0487391614381056
timestep:  4
sample:  0.9728073703149053
timestep:  5
sample:  0.8666058606395146
timestep:  6
sample:  0.5409955448622744
timestep:  7
sample:  1.5718856459733077
timestep:  8
sample:  0.43105586690316
timestep:  9
sample:  0.6717014888624945


In [ ]:
validation_runs = 1
validation_matrix = get_samples_matrix(validation_runs)
v_predicted_output, v_true_output = validate_PCE(validation_matrix, c, alpha)

In [ ]:
# Validation Plot
error = v_true_output - v_predicted_output
lw = 1
fig, ax = plt.subplots(dpi = 150)
# for idx in range(validation_runs):
#     ax.errorbar(list(range(0,timesteps)), v_true_output[idx, :], yerr= np.abs(avg_error), label = "True", color = "green", lw = lw)

for idx in range(validation_runs):
    ax.plot(list(range(0,timesteps)), v_true_output[idx, :], color = "green", label = "True Solution")    
    ax.plot(list(range(0,timesteps)), v_predicted_output[idx, :], label = "PCE Solution", color = "hotpink", lw = lw)


ax.set_xlabel(xlabel = r"Timestep")
ax.set_ylabel(ylabel = r"$RF$ [mW/m2]")

ax.legend(fontsize = 9)

ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

In [ ]:
lw = 1
fig, ax = plt.subplots(dpi = 150)
print("training output shape: ", training_output.shape)
for idx in range(4):
    print(idx)
    ax.plot(list(range(0,timesteps)), training_output[idx, :], label = "True", color = "green", lw = lw)
#for idx in range(mc_runs):    
#    ax.plot(list(range(0,timesteps)), predicted_output[idx, :], label = "PCE", color = "hotpink", ls = "dashed", lw = lw)
    
ax.set_xlabel(xlabel = r"Timestep")
ax.set_ylabel(ylabel = r"Value")

custom_lines = [Line2D([0], [0], color="green", lw=1), Line2D([0], [0], color="hotpink", lw=1)] # 

ax.legend(custom_lines, ['True Solution', 'PCE Solution'], fontsize = 9)

ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

In [ ]:
MSE = np.sum(np.square(training_output[0:100]-predicted_output))/len(training_output)
print("MSE: ", MSE)
lw = 1
fig, ax = plt.subplots(dpi = 150)

ax.plot(list(range(0,timesteps)), training_output.mean(axis=0), label = "True", color = "green", lw = lw)
ax.plot(list(range(0,timesteps)), predicted_output.mean(axis=0), label = "PCE", color = "hotpink", ls = "dashed", lw = lw)

# ax.set_xlim(0,1)
ax.set_xlabel(xlabel = r"Timestep")
ax.set_ylabel(ylabel = r"Value")

custom_lines = [Line2D([0], [0], color="green", lw=1), Line2D([0], [0], color="hotpink", lw=1)] # 

ax.legend(custom_lines, ['True Solution Average', 'PCE Solution Average'], fontsize = 9)

ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

In [9]:
# Use surrogates to compute Sobol analysis
def make_ABC_LS(m, sample_points):
    N_samples_LS = 1000
    stoch_dim = 2
    poly_deg = 2
    
    n_points = sample_points.size
    
    n_params = n_points + 1 
    
    F1 = np.random.normal(mean_Y, sigma_Y, size = (m, 1))
    F2 = np.random.normal(mean_Y, sigma_Y, size = (m, 1))
    
    A = np.hstack((F1, np.random.normal(0, 1, size = (m, n_points))))
    B = np.hstack((F2, np.random.normal(0, 1, size = (m, n_points))))
    
    C = np.zeros((n_params, m , n_params))
    
    for i in range(n_params):
        C[i, :, :] = B
        C[i, :, i] = A[:, i]
        
    y_A = np.zeros(m, dtype = np.float32)
    y_B= np.zeros(m, dtype = np.float32)
    y_C = np.zeros((n_params, m), dtype = np.float32)
    
    c, alpha_set = compute_c(N_samples_LS, stoch_dim, poly_deg)
    
    
    for i in tqdm.tqdm(range(m)):
        y_A[i] = evaluate_LS(c, alpha_set, A[i, :])[50]
        y_B[i] = evaluate_LS(c, alpha_set, B[i, :])[50]
                          
        for j in range(n_params):
            y_C[j, i] = evaluate_LS(c, alpha_set, C[j, i, :])[50]
    
    return y_A, y_B, y_C

In [ ]:
y_A_LS, y_B_LS, y_C_LS = make_ABC_LS(10, sample_points)

In [ ]:
f0sq_LS = get_f_0_sq(sample_points.size, y_A_LS)
S_LS = np.array([get_S_i(sample_points.size, y_A_LS, y_C_LS[i], f0sq_LS) for i in range(102)])
S_T_LS = np.array([get_S_Ti(sample_points.size, y_A_LS, y_B_LS, y_C_LS[i], f0sq_LS) for i in range(102)])

In [ ]:
S1_LS,S_T1_LS = S_LS[0], S_T_LS[0]
S2_LS, S_T2_LS = np.sum(S_LS[1:]), np.sum(S_T_LS[1:])
S12_LS = -0.5 * (S_T1_LS - S1_LS  + S_T2_LS - S2_LS)
main_effect_S_LS = np.array([S1_LS, S2_LS, S12_LS])
tot_effect_ST_LS = np.array([S_T1_LS, S_T2_LS])

In [ ]:
fig, ax = plt.subplots(dpi = 300)
ax.bar(np.arange(0, 3), main_effect_S_LS/main_effect_S_LS.sum(), width = 0.5, alpha = 0.8)
ax.set_xticks(np.arange(0, 3), [r"$S_F$", r"$S_k$", r"$S_{F, k}$"])
ax.set_yticks(np.linspace(0, 1, 11), np.linspace(0, 100, 11, dtype = int))
ax.set_ylabel(ylabel = r"% Contribution to $u(x = 0.5)$")
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

In [ ]:
fig, ax = plt.subplots(dpi = 300)
ax.bar(np.arange(0, 2), tot_effect_ST_LS/tot_effect_ST_LS.sum(), width = 0.5, alpha = 0.8)
ax.set_xticks([0, 1], [r"$S_{T_F}$", r"$S_{T_k}$"])
ax.set_yticks(np.linspace(0, 1, 11), np.linspace(0, 100, 11, dtype = int))
ax.set_ylabel(ylabel = r"% Contribution to $u(x = 0.5)$")
ax.set_ylim(0, 1)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)